# Football Impact Rating — End-to-End Analysis

## Why Goals and Assists Are Not Enough

The Premier League's official statistics page shows you goals, assists, clean sheets. These are terminal events — the final action in a long chain of decisions, movements, and actions that actually determined the outcome. They are the consequence of football, not the cause.

Consider two central midfielders:
- **Player A**: 8 goals, 6 assists in 38 games. The public narrative: excellent season.
- **Player B**: 1 goal, 3 assists in 38 games. The public narrative: quiet season.

Now look at their per-90 data:
- Player A averaged 4.2 progressive passes, won 1.1 tackles, made 18 pressures with 30% success rate
- Player B averaged 7.8 progressive passes, won 2.1 tackles, made 26 pressures with 38% success rate, and had 0.19 xAG/90 compared to Player A's 0.11

Player B was more valuable. Their goals and assists didn't reflect it.

This notebook walks through the Football Impact Rating system that attempts to capture the full picture.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0D1117'

from src.data_generator import FootballDataGenerator
from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.impact_scorer import ImpactScorer
from src.clustering import PlayerArchetypeClusterer
from src.visualizer import FootballVisualizer

print('All modules imported successfully')

## Stage 1: The Dataset

We generate a synthetic dataset of 500 players whose statistical profiles are calibrated against real FBref Premier League 2023-24 data. The key insight in the generation is **position identity as the primary distribution driver** — a generated striker should look nothing like a generated centre-back in the per-90 stats.

We also inject 15 'outlier' players who spike in one specific dimension — think Erling Haaland's xG profile, or Trent Alexander-Arnold's progressive passes. These players should naturally surface as the most interesting cases in the analysis, confirming the system is identifying what it should.

In [ ]:
generator = FootballDataGenerator()
raw_df = generator.generate(n_players=500, seed=42)

print(f'Dataset shape: {raw_df.shape}')
print(f'Positions: {raw_df["position"].value_counts().to_dict()}')
print(f'\nAge distribution:')
print(raw_df.groupby('position')['age'].describe()[['mean', 'min', 'max']].round(1))

In [ ]:
# Compare xG distributions by position — this is the position-specificity check
fig, axes = plt.subplots(1, 4, figsize=(16, 4), facecolor='#0D1117')
outfield_positions = ['ST', 'CM', 'FB', 'CB']
colors = ['#00FF87', '#FFB800', '#FF6B9D', '#00C8FF']

for ax, pos, color in zip(axes, outfield_positions, colors):
    ax.set_facecolor('#0D1117')
    data = raw_df[raw_df['position'] == pos]['xG'].dropna()
    ax.hist(data, bins=20, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(f'{pos}\nxG/90 (mean={data.mean():.3f})', color='white', fontsize=10)
    ax.tick_params(colors='grey')
    for spine in ax.spines.values():
        spine.set_color('grey')
        spine.set_alpha(0.3)

fig.suptitle('xG/90 Distribution by Position — Position Identity Check', 
             color='white', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('Strikers have ~4x the xG rate of CBs — the data is position-authentic')

## Stage 2: Preprocessing — The Minimum Minutes Decision

The 900-minute threshold is not arbitrary. It is the FBref community standard, representing approximately 10 full games. Below this threshold, per-90 numbers become dominated by sample variance: a player who scored in both of his 2 appearances has a 1.0 goals/90 rate that tells us nothing about his true quality.

The winsorisation decision is equally important. **We do not remove outliers — we cap them.** In football analytics, outliers ARE the signal. A 35-xG season is not a data error; it is the most important data point in the dataset. We cap values at the IQR boundary to prevent them from collapsing all other players into a narrow band when we later apply Min-Max scaling.

In [ ]:
preprocessor = DataPreprocessor()
processed_df = preprocessor.run(raw_df, min_minutes=900)

removed = len(raw_df) - len(processed_df)
print(f'Players removed (< 900 mins): {removed}')
print(f'Players retained: {len(processed_df)}')
print(f'\nRetained position breakdown:')
print(processed_df['position'].value_counts())

position_dfs = preprocessor.separate_by_position(processed_df)

## Stage 3: Feature Engineering — From Raw Stats to Football Dimensions

This is where the football intelligence lives. Raw per-90 stats are noisy, correlated, and position-dependent in ways that make direct comparison difficult. Composite metrics aggregate related actions into dimensions that map onto real football concepts.

The composite metrics are:
1. **PPI** — Possession Progression Index: Does the player advance the ball?
2. **DAQ** — Defensive Action Quality: Does the player win the ball back effectively?
3. **CCC** — Chance Creation Contribution: Does the player create dangerous chances?
4. **BRS** — Ball Retention Score: Does the player keep possession under pressure?
5. **PII** — Pressing Intensity Index: Does the player press with purpose?
6. **TGI** — Threat Generation Index: Does the player generate and convert goal threat (net of errors)?

Every weight in every formula has a football reason, not just a statistical one.

In [ ]:
engineer = FeatureEngineer()
featured_dfs = engineer.run(position_dfs)

# Show CM feature distributions — should reflect the CM playing style
cm_features = featured_dfs['CM'][['PPI', 'DAQ', 'CCC', 'BRS', 'PII']]
print('CM Composite Feature Statistics:')
print(cm_features.describe().round(3))

In [ ]:
# Correlation matrix for CM features — are our composite metrics capturing
# independent dimensions or are they just measuring the same thing?
# Low correlations = good; we want orthogonal dimensions.

fig, ax = plt.subplots(figsize=(8, 6), facecolor='#0D1117')
ax.set_facecolor('#0D1117')

corr = cm_features.corr()
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)

features = corr.columns.tolist()
ax.set_xticks(range(len(features)))
ax.set_yticks(range(len(features)))
ax.set_xticklabels(features, color='white', fontsize=10)
ax.set_yticklabels(features, color='white', fontsize=10)

for i in range(len(features)):
    for j in range(len(features)):
        val = corr.iloc[i, j]
        txt = 'black' if -0.5 < val < 0.5 else 'white'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=txt, fontsize=9)

plt.colorbar(im, ax=ax)
fig.suptitle('CM Composite Feature Correlation Matrix\n(Low correlations confirm independent dimensions)', 
             color='white', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## Stage 4: Impact Scoring

The scoring system converts composite metrics into a single 0-100 number using position-specific weights.

**Critical design choice:** Normalisation is done **within each position group**, not globally. A CB's DAQ score of 75 means she's in the top 25% of CBs on defending — not in the top 25% of all players, which would be meaningless because strikers barely tackle. Cross-position normalisation would compress all strikers to near-zero on DAQ, destroying variation.

The weights are the **football opinion** baked into the system. A CB's weights allocate 35% to DAQ (defending is the primary job), 25% to BRS (ball-playing CBs are premium in modern football), 20% to PPI (progressive carrying from the back is a modern tactical asset). Change these weights and you change the analytical philosophy.

In [ ]:
scorer = ImpactScorer()
scored_dfs = scorer.score_all_positions(featured_dfs)

# Summary of score distributions per position
print('Impact Score Distributions by Position:')
print(f'{"Position":<8} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 45)
for pos, df in scored_dfs.items():
    scores = df['impact_score']
    print(f'{pos:<8} {scores.mean():>8.1f} {scores.std():>8.1f} {scores.min():>8.1f} {scores.max():>8.1f}')

## Stage 5: Archetype Clustering

K-Means clustering groups players into football archetypes — the statistical fingerprints of what scouts describe verbally: "box-to-box engine", "deep-lying playmaker", "ball-playing libero".

**Why K-Means?**
- Player feature space is approximately Gaussian per position → K-Means (spherical clusters) is optimal
- We want exactly 4-6 archetypes per position for scouting communication utility
- DBSCAN would mark unusual players as noise — wrong analytically
- Cluster labels are manually assigned by inspecting centroid feature rankings against football knowledge

**Why StandardScaler for clustering but MinMaxScaler for scoring?**
Clustering needs true variance preserved. A feature spanning 0.1-50 should exert more geometric influence than one spanning 0.1-0.8. For scoring, we need bounded 0-100 output that humans understand immediately.

In [ ]:
clusterer = PlayerArchetypeClusterer()
clustered_dfs = clusterer.run(scored_dfs)

# Show CM archetype distribution
cm_archetypes = clustered_dfs['CM']['archetype_label'].value_counts()
print('CM Archetype Distribution:')
print(cm_archetypes.to_string())

print('\nCB Archetype Distribution:')
cb_archetypes = clustered_dfs['CB']['archetype_label'].value_counts()
print(cb_archetypes.to_string())

In [ ]:
# Show mean composite scores per CM archetype — the 'fingerprint' of each archetype
cm_df = clustered_dfs['CM']
norm_cols = [c for c in cm_df.columns if c.endswith('_normalized')]
clean_names = {c: c.replace('_normalized', '') for c in norm_cols}

archetype_profile = cm_df.groupby('archetype_label')[norm_cols].mean() * 100
archetype_profile = archetype_profile.rename(columns=clean_names)
print('CM Archetype Profiles (mean normalized scores, 0-100):')
print(archetype_profile.round(1).to_string())

## Stage 6: Visualisations

The aesthetic is deliberately chosen to mirror the StatsBomb / Opta visual language that football analytics audiences recognise. Dark background, green accents, labels on the players who matter most.

The radar chart is the universal scout report format — a scout in São Paulo uses the same format as one in London. Every scout recognises what a 'dominant' radar looks like.

In [ ]:
viz = FootballVisualizer()

# Generate radar for top CM scorer
top_cm = cm_df.nlargest(1, 'impact_score').iloc[0]['player_name']
print(f'Generating radar for top CM: {top_cm}')
fig = viz.radar_chart(top_cm, cm_df)
plt.show()

In [ ]:
# Position scatter: DAQ vs PPI for all CMs
# The four quadrants tell us:
# High DAQ + High PPI = Complete Midfielder (rare and very valuable)
# High DAQ + Low PPI = Defensive Specialist
# Low DAQ + High PPI = Pure Progresser
# Low DAQ + Low PPI = Rotation Player

fig = viz.position_scatter(cm_df, 'CM', 'DAQ', 'PPI')
plt.show()

In [ ]:
# Archetype heatmap — the statistical fingerprint of each CM archetype
fig = viz.archetype_profile_heatmap(cm_df, 'CM')
plt.show()

In [ ]:
# Top 20 CMs by impact score — annotated with their strongest component
fig = viz.impact_score_distribution(cm_df, 'CM')
plt.show()

## Stage 7: Player Cards

The player card is the system's primary output for individual player assessment. It answers the key scouting question: "What does this player do well, what are they weak at, and where do they sit relative to their positional peers?"

Critically, the card shows the **reasons** behind the score. Not "73/100" but "73/100 because elite BRS and PPI, but below-average DAQ — a ball-player who is a defensive liability."

In [ ]:
all_players = pd.concat(list(clustered_dfs.values()), ignore_index=True)

# Top striker card
top_st = clustered_dfs['ST'].nlargest(1, 'impact_score').iloc[0]['player_name']
st_card = scorer.generate_player_card(top_st, all_players)
print('=== STRIKER PLAYER CARD ===')
for k, v in st_card.items():
    print(f'  {k}: {v}')

In [ ]:
# Top CB card — should look very different from the striker
top_cb = clustered_dfs['CB'].nlargest(1, 'impact_score').iloc[0]['player_name']
cb_card = scorer.generate_player_card(top_cb, all_players)
print('=== CENTRE-BACK PLAYER CARD ===')
for k, v in cb_card.items():
    print(f'  {k}: {v}')

## The Cross-Position Comparison Problem

One of the system's most important outputs is the **warning** it generates when you try to compare players from different positions. This is an analytical safeguard.

A striker scoring 75/100 on PPI and a CM scoring 75/100 on PPI have **not** performed the same progressive actions. The striker's 75 is relative to other strikers (who average lower PPI because progressing the ball is not their primary function); the CM's 75 is relative to other CMs (who are the most active progressors in the squad).

The underlying raw stats are very different. Comparing position-normalised scores across positions is like comparing a winger's tackles to a CDM's: the role context makes the number meaningless.

In [ ]:
# Demonstrate the cross-position warning
top_st_name = clustered_dfs['ST'].nlargest(1, 'impact_score').iloc[0]['player_name']
top_cm_name = clustered_dfs['CM'].nlargest(1, 'impact_score').iloc[0]['player_name']

comparison = scorer.compare_players([top_st_name, top_cm_name], all_players)
print(comparison.to_string())

## Summary: What This System Tells Us That Box Scores Don't

1. **A striker who generates 0.4 xG but makes an error leading to a 0.25 xG counter-attack chance has a net TGI contribution of ~+0.15 on those actions** — meaningful for clubs playing a high-tempo, error-prone style.

2. **A pressing midfielder who presses 30 times per 90 but only wins the ball back 15% of the time contributes far less PII than one who presses 18 times at 35%** — volume without success is statistical noise.

3. **A centre-back's BRS score rewards those who can play under pressure** — the premium modern coaches pay for ball-playing CBs is reflected in the weight structure.

4. **Archetype clustering confirms patterns scouts know intuitively** — the Deep-Lying Playmaker cluster shows high BRS + PPI but low PII, exactly the Busquets/Fabinho statistical fingerprint.

None of this appears in a goals and assists column.